In [1]:
from tifffile import imwrite
from tkinter import filedialog as fd
import os
from os.path import dirname, basename
from scipy.ndimage import gaussian_filter
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rich.progress import track

# matplotlib.use("Agg")
pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
def spots2heatmap(df, scaling_factor, xedges, yedges, tracklength_threshold):
    df_single_frame_spots = df[df["trackID"].isna()]
    img_spots, _, _ = np.histogram2d(
        x=df_single_frame_spots["x"].to_numpy(float) * scaling_factor,
        y=df_single_frame_spots["y"].to_numpy(float) * scaling_factor,
        bins=(xedges, yedges),
    )

    lst_tracklength = []
    # tracks
    df_tracks = df[df["trackID"].notna()]
    all_trackID = df_tracks["trackID"].unique()
    lst_of_arr_x = []
    lst_of_arr_y = []
    for trackID in track(all_trackID, description="Reconstruction"):
        df_current = df_tracks[df_tracks["trackID"] == trackID]
        lst_tracklength.append(df_current.shape[0])
        # for short tracks, treat as spots
        if df_current.shape[0] <= tracklength_threshold:
            lst_of_arr_x.append(df_current["x"].to_numpy(float) * scaling_factor)
            lst_of_arr_y.append(df_current["y"].to_numpy(float) * scaling_factor)
            continue
        # for long tracks, randomly pick tracklength_threshold number of spots
        else:
            chosen_idx = np.random.choice(df_current.shape[0], tracklength_threshold)
            lst_of_arr_x.append(
                df_current.iloc[chosen_idx]["x"].to_numpy(float) * scaling_factor
            )
            lst_of_arr_y.append(
                df_current.iloc[chosen_idx]["y"].to_numpy(float) * scaling_factor
            )
            continue

    img_tracks, _, _ = np.histogram2d(
        x=np.hstack(lst_of_arr_x),
        y=np.hstack(lst_of_arr_y),
        bins=(xedges, yedges),
    )

    heatmap = img_spots + img_tracks

    return heatmap

In [4]:
folder = r"Z:\Bisal_Halder_turbo\PROCESSED_DATA\Intracellular_GEM_tracking\Time_lapse_imaging_cytGEM_NaAsO2_different_strengths\Time_lapse_imaging_cytGEM_NaAsO2_500uM\15min\Acquisition-4"

os.chdir(folder)

fname = (
    "Time_lapse_imaging_cytGEM_NaAsO2_spots_reformatted.csv"
)

time_cutoff = 0  # Keep all detections with t >= 0
tracklength_threshold = 5

um_per_pixel = 0.097
scaling_factor = 2
um_per_pixel_PAINT = um_per_pixel / scaling_factor

# Dimensions of your ROI movie
xpixels = 512
ypixels = 512

xedges = np.arange((xpixels + 1) * scaling_factor)
yedges = np.arange((ypixels + 1) * scaling_factor)

df = pd.read_csv(fname)

required_columns = {"trackID", "x", "y", "t"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}. "
        f"Available columns: {df.columns.tolist()}"
    )

df = df[df["t"] >= time_cutoff]

heatmap = spots2heatmap(
    df,
    scaling_factor,
    xedges,
    yedges,
    tracklength_threshold,
)

output_name = fname[:-4] + "-heatmap.tif"

imwrite(
    output_name,
    np.swapaxes(heatmap, 0, 1),
)

print(f"Saved: {os.path.join(folder, output_name)}")

Output()

Saved: Z:\Bisal_Halder_turbo\PROCESSED_DATA\Intracellular_GEM_tracking\Time_lapse_imaging_cytGEM_NaAsO2_different_strengths\Time_lapse_imaging_cytGEM_NaAsO2_500uM\15min\Acquisition-4\Time_lapse_imaging_cytGEM_NaAsO2_spots_reformatted-heatmap.tif


In [5]:
fname = "cropped-C2-GEM_spots_reformatted.csv"

time_cutoff = 0  # remove the first 200 frames of tracking
tracklength_threshold = 5  # distinguish long versus short tracks

um_per_pixel = 0.117
scaling_factor = 2
um_per_pixel_PAINT = um_per_pixel / scaling_factor
xpixels = 50
ypixels = 50
xedges = np.arange((xpixels + 1) * scaling_factor)
yedges = np.arange((ypixels + 1) * scaling_factor)

df = pd.read_csv(fname)
df = df[df["t"] >= time_cutoff]
heatmap = spots2heatmap(df, scaling_factor, xedges, yedges, tracklength_threshold)
imwrite(fname[:-4] + "-heatmap.tif", np.swapaxes(heatmap, 0, 1))

FileNotFoundError: [Errno 2] No such file or directory: 'cropped-C2-GEM_spots_reformatted.csv'